scilsPeakFilter结果检验

In [1]:
import csv
from pathlib import Path
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from scipy import sparse
import harmonypy as hm

In [12]:
def linreg_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]; y = y[ok]
    n = len(x)
    if n < 2:
        return np.nan, np.nan, np.nan, np.nan

    xm = x.mean()
    ym = y.mean()
    dx = x - xm
    dy = y - ym

    varx = np.sum(dx * dx)
    if varx == 0:
        return np.nan, np.nan, np.nan, np.nan

    slope = np.sum(dx * dy) / varx
    intercept = ym - slope * xm

    yhat = slope * x + intercept
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - ym) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

    # y=0 时 x 截距：x0 = -b/a；取绝对值
    x0_abs = np.abs(-intercept / slope) if slope != 0 else np.nan
    return slope, intercept, r2, x0_abs

In [13]:
def build_linreg_result(df):
    """
    df 的行名格式：
    ctrl1_0_cluster0 / dn1_15_cluster2 这类
    列名为 m/z
    """
    tmp = df.copy()

    # 解析行名：样本号、时间、cluster
    parts = tmp.index.to_series().str.extract(
        r"^([A-Za-z]+\d+)_(\d+)_cluster(\d+)$"
    )

    tmp["_sample"] = parts[0]
    tmp["_time"] = parts[1].astype(int)
    tmp["_cluster"] = parts[2].astype(int)

    # m/z 列
    mz_cols = [c for c in tmp.columns if c not in ["_sample", "_time", "_cluster"]]
    mz_cols = sorted(mz_cols, key=lambda x: float(x))

    result_rows = []

    # 按 sample + cluster 分组
    for (sample, cluster), sub in tmp.groupby(["_sample", "_cluster"], sort=False):
        sub = sub.sort_values("_time")

        # 这里 x 实际上就是 [0,15,30,45,60]
        x = sub["_time"].to_numpy(dtype=float)

        for mz in mz_cols:
            y = sub[mz].to_numpy(dtype=float)
            slope, intercept, r2, x0_abs = linreg_stats(x, y)

            result_rows.append({
                "sample": sample,
                "mz": float(mz),
                "cluster": int(cluster),
                "slope": slope,
                "intercept": intercept,
                "r2": r2,
                "x0_abs": x0_abs
            })

    result_df = pd.DataFrame(result_rows)

    # 排序：先 sample，再 cluster，再 mz
    parts2 = result_df["sample"].str.extract(r"^([A-Za-z]+)(\d+)$")
    result_df["_prefix"] = parts2[0]
    result_df["_sample_num"] = parts2[1].astype(int)

    result_df = result_df.sort_values(
        by=["_prefix", "_sample_num", "cluster", "mz"],
        kind="stable"
    ).drop(columns=["_prefix", "_sample_num"])

    result_df = result_df.reset_index(drop=True)
    return result_df

In [2]:
base_path = Path("/p2/zulab/jtian/data/SA/06_calculateConcentration/CAST_Kmeans_seperate1")
base_path.mkdir(parents=True, exist_ok=True)

ctrl3

In [3]:
GROUP='ctrl3'

In [5]:

data_dir = base_path/GROUP

for csv_file in sorted(data_dir.glob("*.csv"), key=lambda p: int(p.stem)):
    print(f"\nReading file: {csv_file.name}")
    df = pd.read_csv(csv_file, sep=";", comment="#")
    print(df.iloc[:5, :5])
    df.to_csv(csv_file, sep=";", index=False)
    print(f"Cleaned and overwritten: {csv_file.name}")



Reading file: 0.csv
         m/z    Spot 1    Spot 2    Spot 3    Spot 4
0  57.034588  0.151913  0.000000  0.000000  0.110581
1  58.029579  0.184937  0.162343  0.000000  0.000000
2  59.013891  0.799194  0.769255  1.880521  0.000000
3  67.018771  0.355197  0.228945  0.640639  0.000000
4  68.013943  0.000000  0.000000  0.000000  0.488056
Cleaned and overwritten: 0.csv

Reading file: 15.csv
         m/z  Spot 47396  Spot 47397  Spot 47398  Spot 47399
0  57.034588    0.597217    0.000000    0.414708    0.358344
1  58.029579    0.000000    0.000000    0.334957    0.000000
2  59.013891    4.081782    6.295329    4.579338    4.524481
3  67.018771    1.070567    1.158779    0.721309    0.971164
4  68.013943    0.447692    0.000000    1.120953    0.676006
Cleaned and overwritten: 15.csv

Reading file: 30.csv
         m/z  Spot 100805  Spot 100806  Spot 100807  Spot 100808
0  57.034588     0.000000     0.235399     0.000000     0.417222
1  58.029579     0.485109     0.228266     0.995314     1.

In [6]:
ctrl_samples = ["ctrl3"]
time_points  = ["0", "15", "30", "45", "60"]

ctrl_list = []
for sample in ctrl_samples:
    for tp in time_points:
        df = pd.read_csv(data_dir/f'{tp}.csv', sep=";", index_col=0,header=0)
        df_t = df.T.copy()
        df_t = df_t.loc[:, ~df_t.columns.duplicated()]#去掉包里筛出来的重复mz
        df_t.index = [f"{sample}_{tp}_pixel{i+1}" for i in range(df_t.shape[0])]
        ctrl_list.append(df_t)
ctrl_merged = pd.concat(ctrl_list, axis=0)
print("ctrl_merged shape:", ctrl_merged.shape)
print(ctrl_merged.head())
ctrl_merged.to_csv(data_dir/'ctrlIntensity_merged.csv', sep=";")

ctrl_merged shape: (247361, 1178)
m/z             57.034588   58.029579   59.013891   67.018771   68.013943   \
ctrl3_0_pixel1    0.151913    0.184937    0.799194    0.355197    0.000000   
ctrl3_0_pixel2    0.000000    0.162343    0.769255    0.228945    0.000000   
ctrl3_0_pixel3    0.000000    0.000000    1.880521    0.640639    0.000000   
ctrl3_0_pixel4    0.110581    0.000000    0.000000    0.000000    0.488056   
ctrl3_0_pixel5    0.000000    0.000000    1.160304    0.339729    0.418128   

m/z             68.997668   69.034592   70.006122   70.029578   71.013691   \
ctrl3_0_pixel1    0.422714    2.339458    0.000000    0.000000    4.286584   
ctrl3_0_pixel2    0.212294    0.000000    0.366312    0.521995    6.616510   
ctrl3_0_pixel3    0.395135    0.000000    0.000000    0.000000    1.862780   
ctrl3_0_pixel4    0.571333    0.000000    0.000000    0.270308    1.937207   
ctrl3_0_pixel5    0.000000    0.501753    0.243908    0.925107    6.899105   

m/z             ...  888.562

In [7]:
with np.load("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_CAST_Kmeans_seperate/all_clusters.npz") as data:
    ctrl3_cluster = data["ctrl3_cluster"]

In [8]:
ctrl_merged.insert(0, "cluster", ctrl3_cluster)

In [9]:
ctrl_merged

m/z,cluster,57.034588,58.029579,59.013891,67.018771,68.013943,68.997668,69.034592,70.006122,70.029578,...,888.562854,893.490213,918.553689,919.555458,923.615031,944.57995,946.589314,985.604632,987.617316,989.63411
ctrl3_0_pixel1,5,0.151913,0.184937,0.799194,0.355197,0.000000,0.422714,2.339458,0.000000,0.000000,...,0.000000,0.426016,0.267367,0.154390,0.122365,0.265187,0.557014,0.468214,0.045887,0.141272
ctrl3_0_pixel2,5,0.000000,0.162343,0.769255,0.228945,0.000000,0.212294,0.000000,0.366312,0.521995,...,0.563205,0.421467,0.189566,0.426410,0.130137,0.137367,0.351049,0.053420,0.061454,0.041973
ctrl3_0_pixel3,17,0.000000,0.000000,1.880521,0.640639,0.000000,0.395135,0.000000,0.000000,0.000000,...,0.000000,0.370943,0.152571,0.251328,0.098041,0.181843,0.423807,0.290751,0.000000,0.000000
ctrl3_0_pixel4,17,0.110581,0.000000,0.000000,0.000000,0.488056,0.571333,0.000000,0.000000,0.270308,...,0.689285,0.156656,0.210840,0.270308,0.000000,0.270308,0.593176,0.142663,0.064020,0.255291
ctrl3_0_pixel5,17,0.000000,0.000000,1.160304,0.339729,0.418128,0.000000,0.501753,0.243908,0.925107,...,0.000000,0.292214,0.319868,0.107799,0.214565,0.054879,0.705590,0.174220,0.325974,0.169864
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ctrl3_60_pixel50660,17,1.531578,0.290472,16.527840,1.936478,2.081714,2.561432,4.153745,2.662657,7.029415,...,2.149491,1.590993,1.597594,1.573388,1.238327,1.655689,1.395878,1.662144,0.496859,0.734248
ctrl3_60_pixel50661,17,1.538249,0.333884,26.705917,1.136794,1.807211,1.848284,10.283614,2.696242,8.394787,...,3.489083,1.478627,1.957035,1.344478,0.400409,1.947853,1.224240,1.880082,1.221939,1.770775
ctrl3_60_pixel50662,17,1.386536,1.467148,22.594082,2.482866,4.118405,2.047559,11.031020,1.359665,7.945172,...,3.121318,1.072147,2.319706,1.817813,0.746727,1.720272,1.290696,2.404045,2.174841,1.064085
ctrl3_60_pixel50663,17,1.107621,2.643999,11.437082,0.000000,2.620179,3.144215,18.236447,0.000000,14.306178,...,4.401901,3.733755,3.364310,2.333597,0.817082,3.399682,1.986969,3.056876,1.799649,1.222750


In [10]:
ctrl_mz_cols = [c for c in ctrl_merged.columns if c != "cluster"]
ctrl_mz_cols = sorted(ctrl_mz_cols, key=lambda x: float(x))
ctrl_merged = ctrl_merged[["cluster"] + ctrl_mz_cols]

ctrl_tmp = ctrl_merged.copy()
ctrl_tmp["group"] = ctrl_tmp.index.to_series().str.replace(r"_pixel\d+$", "", regex=True)
ctrl_cluster_mean = ctrl_tmp.groupby(
    ["group", "cluster"],
    sort=False
)[ctrl_mz_cols].mean()
ctrl_cluster_mean.index = [
    f"{group}_cluster{cluster}"
    for group, cluster in ctrl_cluster_mean.index
]
ctrl_sort_df = ctrl_cluster_mean.copy()
ctrl_parts = ctrl_sort_df.index.to_series().str.extract(
    r"^ctrl(\d+)_(\d+)_cluster(\d+)$"
)
ctrl_sort_df["_sample_num"] = ctrl_parts[0].astype(int)
ctrl_sort_df["_time"] = ctrl_parts[1].astype(int)
ctrl_sort_df["_cluster"] = ctrl_parts[2].astype(int)
ctrl_sort_df = ctrl_sort_df.sort_values(
    by=["_sample_num", "_cluster", "_time"],
    kind="stable"
)
ctrl_cluster_mean = ctrl_sort_df.drop(
    columns=["_sample_num", "_time", "_cluster"]
)
print(ctrl_cluster_mean.iloc[:10,:5])

m/z                57.034588  58.029579  59.013891  67.018771  68.013943
ctrl3_0_cluster0    0.543266   0.938812   6.932724   1.058000   2.032878
ctrl3_15_cluster0   0.621146   1.002403   8.040421   1.140169   1.833572
ctrl3_30_cluster0   0.667324   1.020731   8.887412   1.236178   1.783366
ctrl3_45_cluster0   0.702297   1.056585   9.403352   1.315716   1.739609
ctrl3_60_cluster0   0.739481   1.103465  10.066621   1.461686   1.727156
ctrl3_0_cluster1    0.514239   0.856209   6.437344   0.999671   2.047905
ctrl3_15_cluster1   0.601896   0.930343   7.637991   1.094351   1.838281
ctrl3_30_cluster1   0.660309   0.944914   8.158509   1.201081   1.596262
ctrl3_45_cluster1   0.665738   1.034584   8.804361   1.369488   1.655657
ctrl3_60_cluster1   0.697307   0.953981   9.584637   1.419895   1.645900


In [14]:
ctrl_linreg_result = build_linreg_result(ctrl_cluster_mean)
print(ctrl_linreg_result.head())

  sample         mz  cluster     slope  intercept        r2      x0_abs
0  ctrl3  57.034588        0  0.003157   0.559987  0.968555  177.367448
1  ctrl3  58.029579        0  0.002557   0.947702  0.973295  370.691045
2  ctrl3  59.013891        0  0.050872   7.139961  0.978614  140.352825
3  ctrl3  67.018771        0  0.006553   1.045766  0.986301  159.591073
4  ctrl3  68.013943        0 -0.004703   1.964397  0.804252  417.716264


In [15]:
ctrl_r2_mean = (
    ctrl_linreg_result
    .groupby(["sample", "mz"], as_index=False)["r2"]
    .mean()
    .rename(columns={"r2": "mean_r2"})
)

ctrl_good = ctrl_r2_mean[ctrl_r2_mean["mean_r2"] >= 0.9].copy()

# 每个样本筛出来多少个代谢物
ctrl_count = (
    ctrl_good
    .groupby("sample", as_index=False)
    .size()
    .rename(columns={"size": "n_metabolites_r2_ge_0.9"})
)

print("ctrl_good:")
print(ctrl_good.head())

print("ctrl_count:")
print(ctrl_count)

ctrl_good:
   sample         mz   mean_r2
0   ctrl3  57.034588  0.914136
2   ctrl3  59.013891  0.959299
3   ctrl3  67.018771  0.960030
9   ctrl3  71.013691  0.922411
10  ctrl3  71.050272  0.943974
ctrl_count:
  sample  n_metabolites_r2_ge_0.9
0  ctrl3                      344


dn3

In [16]:
GROUP='dn3'

In [17]:
data_dir = base_path/GROUP

for csv_file in sorted(data_dir.glob("*.csv"), key=lambda p: int(p.stem)):
    print(f"\nReading file: {csv_file.name}")
    df = pd.read_csv(csv_file, sep=";", comment="#")
    print(df.iloc[:5, :5])
    df.to_csv(csv_file, sep=";", index=False)
    print(f"Cleaned and overwritten: {csv_file.name}")


Reading file: 0.csv
         m/z    Spot 1    Spot 2   Spot 3    Spot 4
0  57.034588  1.215533  0.000000  0.00000  0.000000
1  58.029870  2.885593  3.170958  0.00000  0.482639
2  59.013891  0.000000  4.076946  0.00000  2.448298
3  67.019106  1.519417  3.592274  0.58489  0.000000
4  68.014283  1.314230  0.000000  0.00000  1.382104
Cleaned and overwritten: 0.csv

Reading file: 15.csv
         m/z  Spot 63283  Spot 63284  Spot 63285  Spot 63286
0  57.034588    0.479704    1.238960    0.824226    0.606022
1  58.029870    1.812216    5.114280    1.851414    1.621518
2  59.013891    4.413279    9.617133    4.116673    0.000000
3  67.019106    0.566923    1.636362    1.202924    0.851707
4  68.014283    0.746207    4.314281    2.014774    0.000000
Cleaned and overwritten: 15.csv

Reading file: 30.csv
         m/z  Spot 123867  Spot 123868  Spot 123869  Spot 123870
0  57.034588     0.639176     0.000000     1.331089     0.599654
1  58.029870     0.000000     2.516340     1.762461     0.942314

In [21]:
group_samples = [GROUP]
time_points = ["0", "15", "30", "45", "60"]

group_list = []
for sample in group_samples:
    for tp in time_points:
        df = pd.read_csv(data_dir / f'{tp}.csv', sep=";", index_col=0, header=0)
        df_t = df.T.copy()
        df_t = df_t.loc[:, ~df_t.columns.duplicated()]  # 去掉包里筛出来的重复mz
        df_t.index = [f"{sample}_{tp}_pixel{i+1}" for i in range(df_t.shape[0])]
        group_list.append(df_t)

group_merged = pd.concat(group_list, axis=0)
print("group_merged shape:", group_merged.shape)
print(group_merged.head())

group_merged.to_csv(data_dir / f'{GROUP}Intensity_merged.csv', sep=";")


group_merged shape: (306211, 1146)
m/z           57.034588   58.029870   59.013891   67.019106   68.014283   \
dn3_0_pixel1    1.215533    2.885593    0.000000    1.519417    1.314230   
dn3_0_pixel2    0.000000    3.170958    4.076946    3.592274    0.000000   
dn3_0_pixel3    0.000000    0.000000    0.000000    0.584890    0.000000   
dn3_0_pixel4    0.000000    0.482639    2.448298    0.000000    1.382104   
dn3_0_pixel5    0.467187    0.795948    0.000000    0.254829    0.553703   

m/z           68.998013   69.034592   70.006122   70.029928   71.013691   ...  \
dn3_0_pixel1    2.080432    0.000000    0.971388   28.901642   32.398638  ...   
dn3_0_pixel2    3.962905    6.209502    0.836297   42.337513   29.793065  ...   
dn3_0_pixel3    2.073699    3.041426    0.628215   11.502827    8.708355  ...   
dn3_0_pixel4    1.059014    0.000000    0.000000    6.594609    7.371219  ...   
dn3_0_pixel5    3.086263    1.681872    1.937959   16.881009   13.911779  ...   

m/z           888.567

In [22]:
with np.load("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_CAST_Kmeans_seperate/all_clusters.npz") as data:
    group_cluster = data[f'{GROUP}_cluster']

group_merged.insert(0, "cluster", group_cluster)


In [23]:
group_merged

m/z,cluster,57.034588,58.02987,59.013891,67.019106,68.014283,68.998013,69.034592,70.006122,70.029928,...,888.567297,893.490213,918.553689,919.555458,923.615031,944.57995,946.59878,961.606492,987.622254,989.63411
dn3_0_pixel1,19,1.215533,2.885593,0.000000,1.519417,1.314230,2.080432,0.000000,0.971388,28.901642,...,0.925675,3.669976,0.473123,0.685685,0.157136,0.899962,0.869351,1.118524,0.175932,0.599975
dn3_0_pixel2,19,0.000000,3.170958,4.076946,3.592274,0.000000,3.962905,6.209502,0.836297,42.337513,...,1.003556,4.604383,2.295634,2.574225,0.400725,1.550633,0.896032,1.442612,0.148553,0.644645
dn3_0_pixel3,3,0.000000,0.000000,0.000000,0.584890,0.000000,2.073699,3.041426,0.628215,11.502827,...,0.000000,2.135733,0.889032,0.349309,0.639046,0.216626,0.974816,0.721364,0.205224,0.714865
dn3_0_pixel4,3,0.000000,0.482639,2.448298,0.000000,1.382104,1.059014,0.000000,0.000000,6.594609,...,0.000000,0.260266,0.773978,1.925073,0.131629,0.680083,0.836784,0.730540,0.103918,0.416825
dn3_0_pixel5,3,0.467187,0.795948,0.000000,0.254829,0.553703,3.086263,1.681872,1.937959,16.881009,...,1.401560,2.250990,0.778644,0.583983,0.242245,0.622915,1.401560,0.467187,0.508168,0.380671
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
dn3_60_pixel58565,5,0.777450,1.456999,16.305715,1.433963,1.182492,4.509209,15.127445,1.773737,15.659567,...,0.000000,1.503070,3.207931,3.476208,2.101034,2.122150,1.076912,1.700887,1.410321,0.844637
dn3_60_pixel58566,5,0.000000,1.946036,15.669067,0.582863,1.285774,3.710421,8.194203,0.642887,8.162927,...,3.033731,2.978288,2.195546,1.531200,1.711469,0.912205,1.265916,1.329212,1.580240,0.816640
dn3_60_pixel58567,5,0.579253,1.471479,12.668600,1.317516,1.943463,1.692327,5.222362,2.054518,6.371781,...,0.000000,1.777511,0.779606,1.348277,0.985613,1.985108,0.737723,0.693400,0.703592,1.034200
dn3_60_pixel58568,5,0.331169,0.607144,10.147975,0.958026,1.792520,3.146109,6.218887,3.238100,7.441848,...,2.367861,2.400978,1.561227,1.962375,1.243199,2.291245,1.170920,0.793624,1.212005,1.134781


In [24]:
group_mz_cols = [c for c in group_merged.columns if c != "cluster"]
group_mz_cols = sorted(group_mz_cols, key=lambda x: float(x))
group_merged = group_merged[["cluster"] + group_mz_cols]

group_tmp = group_merged.copy()
group_tmp["group"] = group_tmp.index.to_series().str.replace(r"_pixel\d+$", "", regex=True)

group_cluster_mean = group_tmp.groupby(
    ["group", "cluster"],
    sort=False
)[group_mz_cols].mean()

group_cluster_mean.index = [
    f"{group}_cluster{cluster}"
    for group, cluster in group_cluster_mean.index
]

group_sort_df = group_cluster_mean.copy()
group_parts = group_sort_df.index.to_series().str.extract(
    r"^dn(\d+)_(\d+)_cluster(\d+)$"#####第一部分要改
)

group_sort_df["_sample_num"] = group_parts[0].astype(int)
group_sort_df["_time"] = group_parts[1].astype(int)
group_sort_df["_cluster"] = group_parts[2].astype(int)

group_sort_df = group_sort_df.sort_values(
    by=["_sample_num", "_cluster", "_time"],
    kind="stable"
)

group_cluster_mean = group_sort_df.drop(
    columns=["_sample_num", "_time", "_cluster"]
)

print(group_cluster_mean.iloc[:10, :5])


m/z              57.034588  58.02987  59.013891  67.019106  68.014283
dn3_0_cluster0    0.791823  1.034667  15.646591   1.042390   2.194807
dn3_15_cluster0   0.781026  1.092986  10.674005   1.364844   1.656894
dn3_30_cluster0   0.814015  1.195554  11.412932   1.352603   1.657591
dn3_45_cluster0   0.811837  1.126057  11.444757   1.410817   1.538725
dn3_60_cluster0   0.773876  1.042068  11.508915   1.320254   1.397313
dn3_0_cluster1    0.559094  0.902222   8.071114   0.862840   1.690122
dn3_15_cluster1   0.537755  0.907372   7.132057   0.931447   1.349625
dn3_30_cluster1   0.560807  0.880989   7.873792   1.036434   1.251464
dn3_45_cluster1   0.604334  0.927003   8.648587   1.176692   1.265893
dn3_60_cluster1   0.625227  0.930514   8.686545   1.182387   1.206890


In [25]:
group_linreg_result = build_linreg_result(group_cluster_mean)
print(group_linreg_result.head())


  sample         mz  cluster     slope  intercept        r2        x0_abs
0    dn3  57.034588        0 -0.000034   0.795532  0.001995  23474.644123
1    dn3  58.029870        0  0.000319   1.088692  0.013119   3411.275683
2    dn3  59.013891        0 -0.050031  13.638360  0.355201    272.600040
3    dn3  67.019106        0  0.004011   1.177841  0.420939    293.627287
4    dn3  68.014283        0 -0.011421   2.031697  0.802935    177.890693


In [26]:
group_r2_mean = (
    group_linreg_result
    .groupby(["sample", "mz"], as_index=False)["r2"]
    .mean()
    .rename(columns={"r2": "mean_r2"})
)

group_good = group_r2_mean[group_r2_mean["mean_r2"] >= 0.9].copy()

# 每个样本筛出来多少个代谢物
group_count = (
    group_good
    .groupby("sample", as_index=False)
    .size()
    .rename(columns={"size": "n_metabolites_r2_ge_0.9"})
)

print("group_good:")
print(group_good.head())

print("group_count:")
print(group_count)


group_good:
   sample         mz   mean_r2
10    dn3  71.050272  0.905346
15    dn3  74.024613  0.924866
19    dn3  79.011435  0.929294
49    dn3  96.969588  0.946380
55    dn3  98.973524  0.940041
group_count:
  sample  n_metabolites_r2_ge_0.9
0    dn3                      117
